# VERA 스타일 판단 기준 학습 (Verbalized Learning)

[VERA](https://smcho1201.tistory.com/146) 아이디어를 단순화해서 실습합니다. **모델 가중치는 전혀 학습하지 않고**, 사람이 읽을 수 있는 **자연어 판단 기준(guideline)**을 스스로 개선해 나가는 과정을 직접 구현합니다.

- QLoRA(2번), Vision Adapter(5번) 노트북은 RunPod GPU에서 가중치를 학습했습니다.
- 이 노트북은 **가중치 업데이트가 전혀 없어** 로컬 환경에서 Ollama만으로 실행됩니다.
- 결과물도 벡터가 아니라 사람이 읽고 고칠 수 있는 **문장**이라는 점이 핵심입니다.

## 사전 준비
```bash
uv sync
ollama pull qwen3-vl:4b
```

`1. VLM 학습 데이터 준비.ipynb`를 먼저 실행해 `data/train`, `data/val`이 생성되어 있어야 합니다.


In [ ]:
import vlm_finetune_utils as vu
from PIL import Image


## 1. 초기 판단 기준
사람이 대충 적어본 규칙에서 출발합니다 — 일부러 허점(정상 제품의 인쇄선을 불량으로 오인할 수 있는 표현)을 남겨두었습니다.

In [ ]:
INITIAL_GUIDELINE = "이미지 안에 색이 있는 선이 보이면 불량이다."

PROMPT_TEMPLATE = """당신은 제품 이미지 불량 검사원입니다. 아래 판단 기준에 따라
이 제품 이미지를 검사하세요.

[판단 기준]
{guideline}

다음 JSON 형식으로만 답하세요. 다른 설명은 하지 마세요.
{{"has_defect": true 또는 false, "corner": "좌측 상단"/"우측 상단"/"좌측 하단"/"우측 하단"/null}}
"""


## 2. 평가 함수
주어진 guideline으로 데이터셋 전체를 검사하고 정확도와 틀린 사례를 반환합니다.

In [ ]:
def evaluate(guideline, records, image_root):
    results = []
    for r in records:
        img = Image.open(f"{image_root}/{r['image']}")
        prompt = PROMPT_TEMPLATE.format(guideline=guideline)
        try:
            pred = vu.ask_image_json(img, prompt)
        except Exception:
            pred = {"has_defect": None, "corner": None}

        label = r["label"]
        correct = pred.get("has_defect") == label["has_defect"] and (
            not label["has_defect"] or pred.get("corner") == label["corner"]
        )
        results.append({"record": r, "pred": pred, "correct": correct})

    accuracy = sum(x["correct"] for x in results) / len(results)
    return accuracy, results


## 3. 초기 기준으로 학습셋 평가

In [ ]:
train_records = vu.load_jsonl("data/train/annotations.jsonl")

accuracy, results = evaluate(INITIAL_GUIDELINE, train_records, "data/train")
print(f"초기 정확도: {accuracy:.0%}")


## 4. 틀린 사례를 요약해 판단 기준 재작성
VERA의 핵심 아이디어: **틀린 사례를 모델 스스로에게 보여주고, 더 나은 기준을 자연어로 다시 쓰게** 합니다. 정답 라벨이 없는 실서비스라면 사람 피드백으로 대체할 수 있는 단계입니다.

In [ ]:
def summarize_mistakes(results, max_examples=5):
    mistakes = [r for r in results if not r["correct"]][:max_examples]
    if not mistakes:
        return "(틀린 사례 없음)"

    lines = []
    for m in mistakes:
        label = m["record"]["label"]
        actual = f"불량 ({label['corner']})" if label["has_defect"] else "정상"
        lines.append(f"- 실제: {actual} / 모델 예측: {m['pred']}")
    return "\n".join(lines)


def refine_guideline(guideline, mistakes_text):
    prompt = f"""당신은 제품 불량 검사 기준을 개선하는 역할입니다.

[현재 판단 기준]
{guideline}

[이 기준으로 틀린 사례들]
{mistakes_text}

위 틀린 사례들의 공통 원인을 분석해서, 더 정확하게 불량을 판별할 수 있도록
판단 기준을 2~4문장의 자연어 지침으로 다시 작성해줘. 지침 문장만 출력하고
다른 설명은 하지 마."""
    return vu.ask_text(prompt)


## 5. 여러 라운드 반복
라운드마다 (평가 → 틀린 사례 요약 → 기준 재작성)을 반복하며 기준이 어떻게 바뀌는지 관찰합니다.

In [ ]:
N_ROUNDS = 3
guideline = INITIAL_GUIDELINE
history = []

for round_no in range(1, N_ROUNDS + 1):
    accuracy, results = evaluate(guideline, train_records, "data/train")
    history.append({"round": round_no, "guideline": guideline, "accuracy": accuracy})
    print(f"--- Round {round_no} ---")
    print(f"정확도: {accuracy:.0%}")
    print(f"기준: {guideline}\n")

    mistakes_text = summarize_mistakes(results)
    guideline = refine_guideline(guideline, mistakes_text)

final_guideline = guideline


## 6. 기준 변화와 정확도 추이 확인

In [ ]:
for h in history:
    print(f"[Round {h['round']}] 정확도 {h['accuracy']:.0%}")
    print(f"  {h['guideline']}\n")

print("최종 기준:")
print(final_guideline)


## 7. 검증셋(val)으로 최종 기준 확인
학습에 쓰지 않은 데이터로 최종 판단 기준의 일반화 성능을 확인합니다.

In [ ]:
val_records = vu.load_jsonl("data/val/annotations.jsonl")

initial_val_acc, _ = evaluate(INITIAL_GUIDELINE, val_records, "data/val")
final_val_acc, _ = evaluate(final_guideline, val_records, "data/val")

print(f"초기 기준 - val 정확도: {initial_val_acc:.0%}")
print(f"최종 기준 - val 정확도: {final_val_acc:.0%}")


## 정리
- 이 노트북에서 학습된 것은 **가중치가 아니라 문장**입니다 — `final_guideline`을 그대로 읽고, 사람이 직접 수정해서 다시 사용할 수도 있습니다.
- Prompt Tuning(soft prompt)의 결과물은 사람이 읽을 수 없는 벡터인 반면, VERA류 접근의 결과물은 **해석 가능한 텍스트**라는 차이를 기억하세요.
- GPU/가중치 학습 없이도 일정 수준의 성능 개선이 가능하지만, 표현할 수 있는 패턴이 "프롬프트로 설명 가능한 규칙"으로 제한된다는 한계도 있습니다 — 이 한계를 넘어서려면 다시 QLoRA 같은 가중치 학습이 필요합니다.